In [11]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

In [12]:
def extract_features_from_video(video_path, num_frames=10, resize_shape=(64, 64)):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_idxs = np.linspace(0, total_frames - 1, num_frames).astype(int)
    features = []

    for idx in frame_idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        success, frame = cap.read()
        if success:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            resized = cv2.resize(gray, resize_shape)
            features.append(resized.flatten())
    
    cap.release()
    if features:
        return np.mean(features, axis=0)  # Average the features of selected frames
    else:
        return np.zeros(resize_shape[0] * resize_shape[1])  # Handle empty video

In [17]:
def prepare_dataset(real_dir, fake_dir):
    X, y = [], []
    
    for filename in os.listdir(real_dir):
        if filename.endswith(".mp4"):
            path = os.path.join(real_dir, filename)
            X.append(extract_features_from_video(path))
            y.append(0)  # Label for real

    for filename in os.listdir(fake_dir):
        if filename.endswith(".mp4"):
            path = os.path.join(fake_dir, filename)
            X.append(extract_features_from_video(path))
            y.append(1)  # Label for fake

    return np.array(X), np.array(y)

In [18]:
real_path = r"C:\Users\rimjh\Downloads\dataset\videos_real"
fake_path = r"C:\Users\rimjh\Downloads\dataset\videos_fake"

In [19]:
X, y = prepare_dataset(real_path, fake_path)

In [20]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [21]:
model = SVC(kernel='linear')
model.fit(X_train, y_train)

SVC(kernel='linear')

In [22]:
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["Real", "Fake"]))

Accuracy: 0.36363636363636365
              precision    recall  f1-score   support

        Real       0.33      0.27      0.30        11
        Fake       0.38      0.45      0.42        11

    accuracy                           0.36        22
   macro avg       0.36      0.36      0.36        22
weighted avg       0.36      0.36      0.36        22

